# LLM 12: Token Economics

Hands-on:
1. Build a cost calculator (input / output / cache tokens)
2. Compare model-tier costs across Haiku / Sonnet / Opus / GPT-4o-mini
3. Model the savings of prompt caching
4. Simulate a routing policy
5. Exercises: per-feature cost projection, cache TTL planning

Deps: `pip install tiktoken`  (optional: `anthropic openai`)

## 1. Pricing Table (Illustrative 2026 Snapshot)

In [ ]:
# Prices in USD per 1,000,000 tokens (not 1,000)
PRICES = {
    'claude-haiku':  {'input': 1.00,  'output': 5.00,  'cache_read': 0.10},
    'claude-sonnet': {'input': 3.00,  'output': 15.00, 'cache_read': 0.30},
    'claude-opus':   {'input': 15.00, 'output': 75.00, 'cache_read': 1.50},
    'gpt-4o-mini':   {'input': 0.15,  'output': 0.60,  'cache_read': 0.075},
    'gpt-4o':        {'input': 2.50,  'output': 10.00, 'cache_read': 1.25},
    'gemini-flash':  {'input': 0.075, 'output': 0.30,  'cache_read': 0.019},
}

def cost(model: str, input_tok: int, output_tok: int, cached_tok: int = 0) -> float:
    p = PRICES[model]
    fresh_input = input_tok - cached_tok
    return ((fresh_input * p['input']) + (cached_tok * p['cache_read']) + (output_tok * p['output'])) / 1_000_000

print(f'One call  (1K in, 500 out) on:')
for m in PRICES:
    print(f'  {m:<14} ${cost(m, 1000, 500):.5f}')

## 2. Feature-Level Daily Cost Model

In [ ]:
def daily_feature_cost(model, input_tok_per_call, output_tok_per_call,
                       cached_tok_per_call, calls_per_user, dau):
    c = cost(model, input_tok_per_call, output_tok_per_call, cached_tok_per_call)
    return c * calls_per_user * dau

scenarios = [
    ('MVP (1K DAU, 5 calls)',                 1_000,    5),
    ('Small scale (10K DAU, 10 calls)',      10_000,   10),
    ('Mid-scale (100K DAU, 20 calls)',       100_000,  20),
    ('Large scale (1M DAU, 20 calls)',     1_000_000,  20),
]

print(f"{'scenario':<40} {'Sonnet/day':>12} {'Haiku/day':>12} {'GPT-4o/day':>12} {'Flash/day':>12}")
for name, dau, calls in scenarios:
    s = daily_feature_cost('claude-sonnet', 3000, 400, 0, calls, dau)
    h = daily_feature_cost('claude-haiku',  3000, 400, 0, calls, dau)
    g = daily_feature_cost('gpt-4o',        3000, 400, 0, calls, dau)
    f = daily_feature_cost('gemini-flash',  3000, 400, 0, calls, dau)
    print(f'{name:<40} ${s:>11,.0f} ${h:>11,.0f} ${g:>11,.0f} ${f:>11,.0f}')

## 3. The Caching Payoff

In [ ]:
STATIC_PREFIX = 20_000    # long system+docs
DYNAMIC_SUFFIX = 800      # user turn
OUTPUT = 400
CALLS_PER_DAY = 50_000

no_cache  = cost('claude-sonnet', STATIC_PREFIX + DYNAMIC_SUFFIX, OUTPUT, 0) * CALLS_PER_DAY
with_cache = cost('claude-sonnet', STATIC_PREFIX + DYNAMIC_SUFFIX, OUTPUT, STATIC_PREFIX) * CALLS_PER_DAY

print(f'daily cost no caching: ${no_cache:>9,.2f}')
print(f'daily cost w/ caching: ${with_cache:>9,.2f}')
print(f'monthly savings:       ${(no_cache - with_cache) * 30:>9,.2f}')
print(f'annual savings:        ${(no_cache - with_cache) * 365:>9,.2f}')

## 4. Routing Policy Simulation

In [ ]:
# Suppose 70% of traffic is "simple" (Haiku-solvable),
# 25% is "medium" (Sonnet), 5% is "hard" (Opus).

def one_call(model, kind):
    sizes = {'simple': (1500, 200), 'medium': (4000, 500), 'hard': (8000, 1200)}
    i, o = sizes[kind]
    return cost(model, i, o)

mix = [('simple', 0.70), ('medium', 0.25), ('hard', 0.05)]
calls = 1_000_000

# Policy A: everything through Sonnet
cost_A = sum(one_call('claude-sonnet', k) * frac * calls for k, frac in mix)
# Policy B: route by difficulty
route = {'simple': 'claude-haiku', 'medium': 'claude-sonnet', 'hard': 'claude-opus'}
cost_B = sum(one_call(route[k], k) * frac * calls for k, frac in mix)
# Policy C: everything through Haiku (low quality expected for hard calls)
cost_C = sum(one_call('claude-haiku', k) * frac * calls for k, frac in mix)

print(f'1M calls, mixed difficulty:')
print(f'  A (all Sonnet):        ${cost_A:,.0f}')
print(f'  B (routed by tier):    ${cost_B:,.0f}   <- most realistic')
print(f'  C (all Haiku):         ${cost_C:,.0f}   (but hard calls fail)')

## 5. Exercise: Real Traffic Cost Projection

Take 100 recent production prompts (or stand-ins). For each:
1. Tokenize to get real input length distributions.
2. Simulate output of ~300 tokens (or log real output).
3. Compute p50 / p95 / p99 cost per call for Haiku / Sonnet / Opus.
4. Annualize at your expected traffic.

This is the most useful spreadsheet you'll build this quarter.

## 6. Exercise: Cache TTL Planning

Given:
- 50K static tokens
- Inter-call gap distribution (p50 = 30 s, p95 = 10 min, p99 = 6 hours)
- Cache TTL options (5 min, 1 hour)

For each TTL, compute the fraction of calls that hit the cache, weighted by actual traffic volume. Use that to pick a TTL per feature.

Cache-hit rate × (fresh - cached) × calls × 365 = annual $ you left on the table if you pick the wrong TTL.

## 7. Takeaways

- **Model cost per call before building.**
- **Cache prefixes aggressively** — the single biggest lever.
- **Route by difficulty**: 3–10× savings vs always-top-tier.
- **Output tokens cost more** than input. Enforce conciseness.
- **Track cost as an SLI**, not just an end-of-month invoice surprise.

Next: **13 — Hallucination & Guardrail Basics**, where we start measuring and constraining what the model says.